# Experiment 11: Residual MLP + Feature Engineering + Focal Loss 🚀

## Stacking 3 Improvements on Top of MAPS

### Improvement 1: Residual MLP (Skip Connections)
Our earlier experiments showed deeper MLPs (6-layer, 8-layer) **degraded** performance.
Root cause: vanishing gradients. Fix: **residual connections** (like ResNet).
This lets us go deeper (8+ layers) without degradation.

### Improvement 2: Feature Engineering
MAPS feeds raw marker intensities. But biologists think in **ratios and patterns**:
- Log-transform (protein expression is log-normal)
- Biologically meaningful marker ratios (CD4/CD8, M1/M2 polarity, etc.)
- Statistical features (mean expression, # positive markers)

### Improvement 3: Focal Loss + Label Smoothing
CrossEntropyLoss treats all samples equally. **Focal Loss** focuses on hard cases.
**Label Smoothing** reduces overconfidence and acts as regularization.

---

### Experiment Setup
We train **3 models** and compare:
1. **Baseline**: Exact MAPS MLP (from Experiment 10)
2. **Residual MLP**: 8 residual blocks, same everything else
3. **Full stack**: Residual MLP + Feature Engineering + Focal Loss

**Runtime**: ~45-60 min on Kaggle P100

In [ ]:
import os
import sys
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, SequentialSampler

from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    classification_report, confusion_matrix
)

print(f"PyTorch: {torch.__version__}")
print(f"Device: {'GPU (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'CPU'}")

## 1. Load and Prepare Data (Same as Experiment 10)

In [ ]:
# Load annotation CSV
df = pd.read_csv("/kaggle/input/chl-codex-annotated/cHL_CODEX_annotation.csv")

# Column definitions
MARKER_COLS = [
    'BCL.2', 'CCR6', 'CD11b', 'CD11c', 'CD15', 'CD16', 'CD162', 'CD163',
    'CD2', 'CD20', 'CD206', 'CD25', 'CD30', 'CD31', 'CD4', 'CD44',
    'CD45', 'CD45RA', 'CD45RO', 'CD5', 'CD56', 'CD57', 'CD68', 'CD69',
    'CD7', 'CD8', 'Collagen.4', 'Cytokeratin', 'DAPI.01', 'EGFR',
    'FoxP3', 'Granzyme.B', 'HLA.DR', 'IDO.1', 'LAG.3', 'MCT', 'MMP.9',
    'MUC.1', 'PD.1', 'PD.L1', 'Podoplanin', 'T.bet', 'TCR.g.d', 'TCRb',
    'Tim.3', 'VISA', 'Vimentin', 'a.SMA', 'b.Catenin'
]
FEATURE_COLS = MARKER_COLS + ['cellSize']  # 50 features (same as MAPS)

CLASS_NAMES = ['B', 'CD4', 'CD8', 'DC', 'Endothelial', 'Epithelial',
               'Lymphatic', 'M1', 'M2', 'Mast', 'Monocyte', 'NK',
               'Neutrophil', 'Other', 'TReg', 'Tumor']
LABEL_MAP = {name: i for i, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)

# Filter and extract
df = df[df['cellType'].isin(CLASS_NAMES)].reset_index(drop=True)
X_raw = df[FEATURE_COLS].values.astype(np.float64)
y = df['cellType'].map(LABEL_MAP).values.astype(np.int64)

# Same train/valid split as Experiment 10
np.random.seed(42)
perm = np.random.permutation(len(X_raw))
n_train = int(0.8 * len(X_raw))
train_idx, valid_idx = perm[:n_train], perm[n_train:]

X_train_raw, y_train = X_raw[train_idx], y[train_idx]
X_valid_raw, y_valid = X_raw[valid_idx], y[valid_idx]

print(f"Dataset: {len(df):,} cells | Train: {len(X_train_raw):,} | Valid: {len(X_valid_raw):,}")
print(f"Base features: {len(FEATURE_COLS)} (49 markers + cellSize)")

## 2. Feature Engineering 🧬

We add biologically meaningful features ON TOP of the raw 50 features.

In [ ]:
def engineer_features(X, marker_cols, feature_cols):
    """
    Create biologically meaningful features from protein marker expressions.
    Input X has columns in the order of feature_cols (markers + cellSize).
    Returns augmented feature matrix.
    """
    # Build a column index lookup
    col_idx = {name: i for i, name in enumerate(feature_cols)}
    
    # Helper to safely get marker ratios
    def ratio(a, b, eps=1e-6):
        return X[:, col_idx[a]] / (X[:, col_idx[a]] + X[:, col_idx[b]] + eps)
    
    new_features = []
    new_names = []
    
    # --- Log-transformed marker values ---
    X_markers = X[:, :len(marker_cols)]
    X_log = np.log1p(X_markers)  # log(1+x), handles zeros gracefully
    new_features.append(X_log)
    new_names.extend([f'log_{m}' for m in marker_cols])
    
    # --- Biologically meaningful marker ratios ---
    # T cell subtype: CD4 vs CD8
    new_features.append(ratio('CD4', 'CD8').reshape(-1, 1))
    new_names.append('ratio_CD4_CD8')
    
    # Macrophage polarity: M1 (CD68) vs M2 (CD163)
    new_features.append(ratio('CD68', 'CD163').reshape(-1, 1))
    new_names.append('ratio_M1_M2')
    
    # Immune checkpoint: PD1 vs PDL1
    new_features.append(ratio('PD.1', 'PD.L1').reshape(-1, 1))
    new_names.append('ratio_PD1_PDL1')
    
    # B vs T cell:
    cd20 = X[:, col_idx['CD20']]
    cd4 = X[:, col_idx['CD4']]
    cd8 = X[:, col_idx['CD8']]
    new_features.append((cd20 / (cd20 + cd4 + cd8 + 1e-6)).reshape(-1, 1))
    new_names.append('ratio_B_vs_T')
    
    # Naive vs Memory T cell: CD45RA / (CD45RA + CD45RO)
    new_features.append(ratio('CD45RA', 'CD45RO').reshape(-1, 1))
    new_names.append('ratio_naive_memory')
    
    # NK indicator: CD56 * CD7
    new_features.append((X[:, col_idx['CD56']] * X[:, col_idx['CD7']]).reshape(-1, 1))
    new_names.append('interact_NK')
    
    # Cytotoxic T: CD8 * Granzyme.B
    new_features.append((X[:, col_idx['CD8']] * X[:, col_idx['Granzyme.B']]).reshape(-1, 1))
    new_names.append('interact_cytotoxic')
    
    # Treg indicator: CD4 * FoxP3
    new_features.append((X[:, col_idx['CD4']] * X[:, col_idx['FoxP3']]).reshape(-1, 1))
    new_names.append('interact_treg')
    
    # DC activation: HLA.DR * CD11c
    new_features.append((X[:, col_idx['HLA.DR']] * X[:, col_idx['CD11c']]).reshape(-1, 1))
    new_names.append('interact_DC')
    
    # --- Statistical features per cell ---
    # Mean marker expression
    new_features.append(X_markers.mean(axis=1, keepdims=True))
    new_names.append('stat_mean_expr')
    
    # Std of marker expression
    new_features.append(X_markers.std(axis=1, keepdims=True))
    new_names.append('stat_std_expr')
    
    # Max marker value
    new_features.append(X_markers.max(axis=1, keepdims=True))
    new_names.append('stat_max_expr')
    
    # Number of "positive" markers (expression > 0.1)
    new_features.append((X_markers > 0.1).sum(axis=1, keepdims=True).astype(np.float64))
    new_names.append('stat_n_positive')
    
    # Combine: original features + engineered features
    X_augmented = np.concatenate([X] + new_features, axis=1)
    all_names = list(feature_cols) + new_names
    
    return X_augmented, all_names


# Apply feature engineering
X_train_eng, eng_feature_names = engineer_features(X_train_raw, MARKER_COLS, FEATURE_COLS)
X_valid_eng, _ = engineer_features(X_valid_raw, MARKER_COLS, FEATURE_COLS)

NUM_FEATURES_BASE = len(FEATURE_COLS)  # 50
NUM_FEATURES_ENG = X_train_eng.shape[1]  # 50 + engineered

print(f"Base features: {NUM_FEATURES_BASE}")
print(f"Engineered features: {NUM_FEATURES_ENG} (+{NUM_FEATURES_ENG - NUM_FEATURES_BASE} new)")
print(f"\nNew features added:")
for name in eng_feature_names[NUM_FEATURES_BASE:]:
    print(f"  • {name}")

## 3. Model Architectures

In [ ]:
# ====================================================================
# MODEL 1: Exact MAPS MLP (baseline)
# ====================================================================
class MLP_Baseline(nn.Module):
    """Exact MAPS 4-layer MLP."""
    def __init__(self, input_dim=50, hidden_dim=512, num_classes=16, dropout=0.10):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(p=dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Dropout(p=dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Dropout(p=dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Dropout(p=dropout)
        )
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, batch):
        features = self.fc(batch)
        logits = self.classifier(features)
        probs = torch.softmax(logits, dim=-1)
        return logits, probs


# ====================================================================
# MODEL 2: Residual MLP
# ====================================================================
class ResidualBlock(nn.Module):
    """A single residual block: Linear→BN→ReLU→Dropout + skip connection."""
    def __init__(self, dim, dropout=0.10):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.ReLU(),
            nn.Dropout(p=dropout),
        )
    
    def forward(self, x):
        return x + self.block(x)  # Skip connection!


class ResidualMLP(nn.Module):
    """
    MLP with residual (skip) connections.
    Unlike plain deep MLPs that degrade, this can go 8-12 layers deep.
    """
    def __init__(self, input_dim, hidden_dim=512, num_classes=16,
                 num_blocks=8, dropout=0.10):
        super().__init__()
        # Project input to hidden dimension
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout)
        )
        # Stack of residual blocks
        self.blocks = nn.ModuleList([
            ResidualBlock(hidden_dim, dropout) for _ in range(num_blocks)
        ])
        self.classifier = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, batch):
        x = self.input_proj(batch)
        for block in self.blocks:
            x = block(x)
        logits = self.classifier(x)
        probs = torch.softmax(logits, dim=-1)
        return logits, probs


# Print architectures
print("Model 1 (Baseline): Exact MAPS 4-layer MLP")
print(f"  50 → 512 → 512 → 512 → 512 → 16")
print(f"  Params: {sum(p.numel() for p in MLP_Baseline(50).parameters()):,}")

print(f"\nModel 2 (ResidualMLP): 8 residual blocks + BatchNorm")
print(f"  {NUM_FEATURES_ENG} → 512 → [Res×8] → 16")
res_test = ResidualMLP(NUM_FEATURES_ENG, num_blocks=8)
print(f"  Params: {sum(p.numel() for p in res_test.parameters()):,}")

## 4. Loss Functions

In [ ]:
class FocalLoss(nn.Module):
    """
    Focal Loss (Lin et al., 2017)
    Down-weights easy examples, focuses on hard/misclassified ones.
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    """
    def __init__(self, gamma=2.0, label_smoothing=0.05, weight=None):
        super().__init__()
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        self.weight = weight  # class weights
    
    def forward(self, inputs, targets):
        # Apply label smoothing via cross-entropy
        ce_loss = F.cross_entropy(inputs, targets, weight=self.weight,
                                 label_smoothing=self.label_smoothing,
                                 reduction='none')
        # Focal modulation
        pt = torch.exp(-ce_loss)  # probability of correct class
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()


print("Loss functions:")
print("  1. CrossEntropyLoss (MAPS baseline)")
print("  2. FocalLoss(gamma=2.0, label_smoothing=0.05)")

## 5. Training Infrastructure

In [ ]:
class CellDataset(Dataset):
    """Dataset with MAPS-style normalization."""
    def __init__(self, X, y, is_train=True, mean=None, std=None, divide_255=True):
        self.y = y
        self.divide_255 = divide_255
        
        if is_train:
            self.mean = np.mean(X, axis=0)
            self.std = np.std(X, axis=0)
        else:
            self.mean = mean
            self.std = std
        
        self.x = (X - self.mean) / (self.std + 1e-12)
    
    def __len__(self):
        return self.x.shape[0]
    
    def __getitem__(self, idx):
        feature = self.x[idx] / 255.0 if self.divide_255 else self.x[idx]
        return feature, int(self.y[idx])


def make_weighted_sampler(labels):
    labels_list = labels.tolist()
    n = float(len(labels_list))
    unique_labels = sorted(set(labels_list))
    weight_per_class = {c: n / labels_list.count(c) for c in unique_labels}
    sample_weights = [weight_per_class[l] for l in labels_list]
    return WeightedRandomSampler(sample_weights, len(sample_weights))


DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 7325111
BATCH_SIZE = 128
LR = 0.001
MAX_EPOCHS = 500
MIN_EPOCHS = 250
PATIENCE = 100


def set_seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def train_epoch(model, loader, optimizer, loss_fn, device, num_classes):
    model.train()
    total_loss = 0
    all_labels, all_preds = [], []
    for features, labels in loader:
        features, labels = features.to(device), labels.to(device)
        logits, probs = model(features)
        optimizer.zero_grad()
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        all_labels.extend(labels.cpu().numpy().tolist())
        all_preds.extend(torch.argmax(probs, dim=1).cpu().numpy().tolist())
    acc = accuracy_score(all_labels, all_preds)
    return total_loss / len(loader), acc


def valid_epoch(model, loader, loss_fn, device, num_classes):
    model.eval()
    total_loss = 0
    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for features, labels in loader:
            features, labels = features.to(device), labels.to(device)
            logits, probs = model(features)
            loss = loss_fn(logits, labels)
            total_loss += loss.item()
            all_labels.extend(labels.cpu().numpy().tolist())
            all_preds.extend(torch.argmax(probs, dim=1).cpu().numpy().tolist())
            all_probs.append(probs.cpu().numpy())
    all_probs = np.concatenate(all_probs, axis=0)
    acc = accuracy_score(all_labels, all_preds)
    return total_loss / len(loader), acc, np.array(all_labels), np.array(all_preds), all_probs


def train_model(model, train_loader, valid_loader, loss_fn, lr=LR,
                max_epochs=MAX_EPOCHS, min_epochs=MIN_EPOCHS,
                patience=PATIENCE, device=DEVICE, label="Model"):
    """
    Full training pipeline. Returns best model state and history.
    """
    optimizer = optim.Adam(model.parameters(), lr=lr)
    best_loss = float('inf')
    counter = 0
    best_state = None
    history = {'train_loss': [], 'valid_loss': [], 'train_acc': [], 'valid_acc': []}
    
    print(f"\n{'='*70}")
    print(f"Training {label}")
    print(f"{'='*70}")
    
    for epoch in range(max_epochs):
        t0 = time.time()
        tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, loss_fn, device, NUM_CLASSES)
        vl_loss, vl_acc, _, _, _ = valid_epoch(model, valid_loader, loss_fn, device, NUM_CLASSES)
        
        history['train_loss'].append(tr_loss)
        history['valid_loss'].append(vl_loss)
        history['train_acc'].append(tr_acc)
        history['valid_acc'].append(vl_acc)
        
        if vl_loss < best_loss:
            best_loss = vl_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            counter = 0
            marker = ' ← BEST'
        else:
            counter += 1
            marker = ''
        
        if epoch % 20 == 0 or counter == 0:
            print(f"  Epoch {epoch:3d} | TrLoss={tr_loss:.4f} TrAcc={tr_acc:.4f} | "
                  f"VlLoss={vl_loss:.4f} VlAcc={vl_acc:.4f} | {time.time()-t0:.1f}s{marker}")
        
        if counter > patience and epoch >= min_epochs:
            print(f"  Early stopping at epoch {epoch}")
            break
    
    model.load_state_dict(best_state)
    return model, history

print("✅ Training infrastructure ready")

## 6. Train Model A: MAPS Baseline (50 features, CrossEntropy)

In [ ]:
# Model A: Exact MAPS baseline (for comparison)
set_seed(SEED)

# Dataset with base 50 features
ds_train_A = CellDataset(X_train_raw, y_train, is_train=True)
ds_valid_A = CellDataset(X_valid_raw, y_valid, is_train=False,
                         mean=ds_train_A.mean, std=ds_train_A.std)

loader_train_A = DataLoader(ds_train_A, batch_size=BATCH_SIZE,
                            sampler=make_weighted_sampler(y_train),
                            drop_last=True, num_workers=2)
loader_valid_A = DataLoader(ds_valid_A, batch_size=BATCH_SIZE,
                            sampler=SequentialSampler(ds_valid_A),
                            drop_last=False, num_workers=2)

model_A = MLP_Baseline(input_dim=NUM_FEATURES_BASE, hidden_dim=512,
                       num_classes=NUM_CLASSES, dropout=0.10)
model_A.to(DEVICE, dtype=torch.float64)

model_A, history_A = train_model(
    model_A, loader_train_A, loader_valid_A,
    loss_fn=nn.CrossEntropyLoss(),
    label="Model A: MAPS Baseline (50 features, CE Loss)"
)

## 7. Train Model B: Residual MLP (50 features, CrossEntropy)

In [ ]:
# Model B: Residual MLP with base features
set_seed(SEED)

model_B = ResidualMLP(input_dim=NUM_FEATURES_BASE, hidden_dim=512,
                      num_classes=NUM_CLASSES, num_blocks=8, dropout=0.10)
model_B.to(DEVICE, dtype=torch.float64)

model_B, history_B = train_model(
    model_B, loader_train_A, loader_valid_A,  # Same data as baseline
    loss_fn=nn.CrossEntropyLoss(),
    label="Model B: Residual MLP (50 features, CE Loss, 8 blocks)"
)

## 8. Train Model C: Full Stack (Engineered Features + ResidualMLP + Focal Loss)

In [ ]:
# Model C: Everything combined
set_seed(SEED)

# Dataset with engineered features
ds_train_C = CellDataset(X_train_eng, y_train, is_train=True)
ds_valid_C = CellDataset(X_valid_eng, y_valid, is_train=False,
                         mean=ds_train_C.mean, std=ds_train_C.std)

loader_train_C = DataLoader(ds_train_C, batch_size=BATCH_SIZE,
                            sampler=make_weighted_sampler(y_train),
                            drop_last=True, num_workers=2)
loader_valid_C = DataLoader(ds_valid_C, batch_size=BATCH_SIZE,
                            sampler=SequentialSampler(ds_valid_C),
                            drop_last=False, num_workers=2)

# Compute class weights for Focal Loss
class_counts = np.bincount(y_train)
class_weights = torch.tensor(
    len(y_train) / (NUM_CLASSES * class_counts), dtype=torch.float64
).to(DEVICE)

model_C = ResidualMLP(input_dim=NUM_FEATURES_ENG, hidden_dim=512,
                      num_classes=NUM_CLASSES, num_blocks=8, dropout=0.10)
model_C.to(DEVICE, dtype=torch.float64)

focal_loss = FocalLoss(gamma=2.0, label_smoothing=0.05, weight=class_weights)

model_C, history_C = train_model(
    model_C, loader_train_C, loader_valid_C,
    loss_fn=focal_loss,
    label=f"Model C: ResidualMLP + {NUM_FEATURES_ENG} features + Focal Loss"
)

## 9. Comparison & Analysis

In [ ]:
# Evaluate all models
results = {}

eval_ce = nn.CrossEntropyLoss()  # Use CE for consistent eval across all models

for name, model, loader in [
    ('A: MAPS Baseline', model_A, loader_valid_A),
    ('B: ResidualMLP', model_B, loader_valid_A),
    ('C: Full Stack', model_C, loader_valid_C),
]:
    _, _, gt, pred, probs = valid_epoch(model, loader, eval_ce, DEVICE, NUM_CLASSES)
    w_f1 = f1_score(gt, pred, average='weighted')
    m_f1 = f1_score(gt, pred, average='macro')
    acc = accuracy_score(gt, pred)
    results[name] = {'weighted_f1': w_f1, 'macro_f1': m_f1, 'accuracy': acc,
                     'gt': gt, 'pred': pred, 'probs': probs}
    print(f"{name:30s} | W-F1={w_f1:.4f} | M-F1={m_f1:.4f} | Acc={acc:.4f}")

print("\n" + "=" * 70)
print("MAPS target: 0.9000 weighted F1")
best_name = max(results, key=lambda k: results[k]['weighted_f1'])
best_f1 = results[best_name]['weighted_f1']
print(f"\nBest model: {best_name} with F1={best_f1:.4f}")
if best_f1 >= 0.90:
    print("🎉 TARGET ACHIEVED!")

In [ ]:
# Per-class comparison
print("\nPer-class F1 comparison:")
print(f"{'Class':>15s}  {'A: Baseline':>12s}  {'B: Residual':>12s}  {'C: Full':>12s}  {'Winner':>8s}")
print("-" * 70)

for i, name in enumerate(CLASS_NAMES):
    f1s = {}
    for model_name in results:
        per_f1 = f1_score(results[model_name]['gt'], results[model_name]['pred'], average=None)
        f1s[model_name] = per_f1[i]
    
    winner = max(f1s, key=f1s.get)
    vals = [f1s[k] for k in results]
    marker = '→' if max(vals) - min(vals) > 0.02 else ' '
    print(f"{name:>15s}  " + "  ".join(f"{v:12.4f}" for v in vals) + f"  {marker}{winner.split(':')[0]}")

In [ ]:
# Full classification report for best model
print(f"\nDetailed report for best model ({best_name}):")
print("=" * 70)
print(classification_report(results[best_name]['gt'], results[best_name]['pred'],
                            target_names=CLASS_NAMES, digits=4))

In [ ]:
# Confusion matrix for best model
fig, ax = plt.subplots(figsize=(14, 12))
cm = confusion_matrix(results[best_name]['gt'], results[best_name]['pred'])
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title(f'Best Model: {best_name} (W-F1={best_f1:.4f})', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Training curves comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for name, hist, color in [
    ('A: Baseline', history_A, 'blue'),
    ('B: Residual', history_B, 'green'),
    ('C: Full Stack', history_C, 'red'),
]:
    axes[0].plot(hist['valid_loss'], label=name, color=color, alpha=0.7)
    axes[1].plot(hist['valid_acc'], label=name, color=color, alpha=0.7)

axes[0].set_title('Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Training Curves Comparison', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ====================================================================
# SUMMARY
# ====================================================================
print("\n" + "=" * 70)
print("EXPERIMENT 11: RESIDUAL MLP + FEATURE ENGINEERING - SUMMARY")
print("=" * 70)

print("\n  Models trained:")
print("  A) MAPS Baseline: 4-layer MLP, 50 features, CrossEntropy")
print("  B) Residual MLP:  8 residual blocks, 50 features, CrossEntropy")
print(f"  C) Full Stack:    8 residual blocks, {NUM_FEATURES_ENG} features, Focal Loss")

print("\n  Results:")
for name in results:
    r = results[name]
    print(f"    {name:30s}: W-F1={r['weighted_f1']:.4f}  M-F1={r['macro_f1']:.4f}")

print(f"\n  Best: {best_name} (W-F1={best_f1:.4f})")
print(f"  vs MAPS target (0.90): {'+' if best_f1 >= 0.90 else ''}{(best_f1-0.90)*100:.1f}pp")